## Mapping — stations and counties

Exploratory maps: spring means (e.g. tmax) by station, with optional radius filter. Uses `data/merged.csv` and the same county shapefile as in data creation.

**Requires:** `data/merged.csv` (from `dawn_data_creation.ipynb`), and `dawn_config.COUNTY_SHP_PATH` pointing to the county shapefile.

### 1. Config and load data

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
from geopy.distance import geodesic

from dawn_config import DATA_DIR, COUNTY_SHP_PATH, CORN_BELT_STATES

merged = pd.read_csv(DATA_DIR / "merged.csv")
all_counties = gpd.read_file(COUNTY_SHP_PATH).to_crs("EPSG:4326")

years = sorted(merged["year"].unique())
print(f"merged: shape={merged.shape}, years={years[0]}..{years[-1]}, #counties={merged['county_name'].nunique()}")

### 2. Spring means (for plotting)

In [ ]:
spring = merged[merged["month"].isin([3, 4, 5])]
spring_means = spring.groupby("id")[["tmax", "tmin", "prcp"]].mean().reset_index()
spring_means.columns = ["id", "spring_tmax_mean", "spring_tmin_mean", "spring_prcp_mean"]
merged = merged.sort_values(["id", "date"]).merge(spring_means, on="id", how="left")

### 3. Helpers: filter by radius, state centroid, plot

In [ ]:
def filter_stations_within_radius(df, center_coords, radius_km):
    stations = df[["id", "latitude", "longitude"]].drop_duplicates()
    valid_ids = stations[
        stations.apply(
            lambda r: geodesic((r["latitude"], r["longitude"]), center_coords).km <= radius_km,
            axis=1,
        )
    ]["id"]
    return df[df["id"].isin(valid_ids)]

def get_state_centroid(state_fips, counties_gdf):
    state_counties = counties_gdf[counties_gdf["STATEFP"] == state_fips]
    centroid = state_counties.unary_union.centroid
    return (centroid.y, centroid.x)

def plot_stations_with_tmax(df, counties_gdf, title, state_fips=None, ax_size=(10, 10), center_coords=None, radius_km=None):
    from shapely.geometry import Point as ShapelyPoint
    if state_fips:
        plot_counties = counties_gdf[counties_gdf["STATEFP"] == state_fips]
        state_borders = None
    else:
        plot_counties = counties_gdf
        state_borders = counties_gdf.dissolve(by="STATEFP")
    station_df = df.drop_duplicates(subset="id").copy()
    station_df["geometry"] = station_df.apply(
        lambda r: Point(r["longitude"], r["latitude"]), axis=1
    )
    station_gdf = gpd.GeoDataFrame(station_df, geometry="geometry", crs="EPSG:4326")
    if center_coords and radius_km:
        station_gdf = station_gdf[
            station_gdf.apply(
                lambda r: geodesic((r["latitude"], r["longitude"]), center_coords).km <= radius_km,
                axis=1,
            )
        ]
    fig, ax = plt.subplots(figsize=ax_size)
    plot_counties.plot(ax=ax, color="white", edgecolor="gray")
    if state_borders is not None:
        state_borders.boundary.plot(ax=ax, edgecolor="black", linewidth=1)
    ax.set_aspect("equal")
    if station_gdf.empty:
        ax.set_title(title + " (no stations)")
        plt.axis("off")
        plt.tight_layout()
        plt.show()
        return
    station_gdf.plot(ax=ax, column="spring_tmax_mean", cmap="coolwarm", markersize=40, legend=True,
                     legend_kwds={"label": "Spring mean tmax (°C)", "shrink": 0.6})
    if center_coords and radius_km:
        proj = "EPSG:5070"
        center_geom = gpd.GeoSeries([ShapelyPoint(center_coords[1], center_coords[0])], crs="EPSG:4326").to_crs(proj).geometry[0]
        circle = gpd.GeoSeries([center_geom.buffer(radius_km * 1000)], crs=proj).to_crs("EPSG:4326")
        circle.boundary.plot(ax=ax, edgecolor="black", linestyle="--", linewidth=2)
    ax.set_title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

<p align="center">
  <img src="visualizations/MI.png" width=600/>
</p>

<p align="center">
  <img src="visualizations/IA.png" width="600"/>
</p>

<p align="center">
  <img src="visualizations/IL.png" width=600/>
</p>

In [ ]:
<p align="center">
  <img src="visualizations/MN.png" width=600/>
</p>

<p align="center">
  <img src="visualizations/KS.png" width=600/>
</p>

<p align="center">
  <img src="visualizations/IN.png" width=600/>
</p>

<p align="center">
  <img src="visualizations/MO.png" width=600/>
</p>

<p align="center">
  <img src="visualizations/NE.png" width=600/>
</p>

<p align="center">
  <img src="visualizations/OH.png" width=600/>
</p>

<p align="center">
  <img src="visualizations/WI.png" width=600/>
</p>

### 4. Plot by state (radius optional)

In [ ]:
RADIUS_KM = 100  # set to None to plot all stations in the state

for state_abbr, fips in CORN_BELT_STATES.items():
    state_data = merged[merged["state"] == state_abbr]
    center = get_state_centroid(fips, all_counties)
    plot_stations_with_tmax(
        state_data,
        all_counties,
        title=f"{state_abbr} — Spring mean tmax",
        state_fips=fips,
        ax_size=(10, 10),
        center_coords=center if RADIUS_KM else None,
        radius_km=RADIUS_KM,
    )

### 5. Plot full Corn Belt (all states)

In [ ]:
corn_belt_fips = set(CORN_BELT_STATES.values())
corn_belt_counties = all_counties[all_counties["STATEFP"].isin(corn_belt_fips)]
corn_belt_data = merged[merged["state"].isin(CORN_BELT_STATES.keys())]

plot_stations_with_tmax(
    corn_belt_data,
    corn_belt_counties,
    title="Corn Belt — Spring mean tmax",
    state_fips=None,
    ax_size=(14, 10),
    center_coords=None,
    radius_km=None,
)

<p align="center">
  <img src="visualizations/all_states.png" width="600"/>
</p>

### 6. Cluster map (state groups)

In [ ]:
from matplotlib.colors import ListedColormap

CLUSTER_DEFS = {0: ["MI", "MN", "WI"], 1: ["IN", "KS", "MO", "NE", "OH"], 2: ["IA", "IL"]}
FIPS_TO_ABBR = {v: k for k, v in CORN_BELT_STATES.items()}

def plot_counties_by_cluster(counties_gdf, title, ax_size=(12, 8)):
    gdf = counties_gdf.copy()
    gdf["state_abbr"] = gdf["STATEFP"].map(FIPS_TO_ABBR)
    def assign(s):
        for c, states in CLUSTER_DEFS.items():
            if s in states:
                return c
        return None
    gdf["cluster"] = gdf["state_abbr"].map(assign)
    borders = gdf.dissolve(by="STATEFP")
    fig, ax = plt.subplots(figsize=ax_size)
    gdf.plot(ax=ax, column="cluster", categorical=True,
             cmap=ListedColormap(["#FFC6C2", "#C3E0DD", "#FAE9DA"]),
             edgecolor="gray", missing_kwds={"color": "lightgrey"})
    borders.boundary.plot(ax=ax, edgecolor="black", linewidth=1)
    ax.set_title(title)
    ax.axis("off")
    ax.set_aspect("auto")
    plt.tight_layout()
    plt.show()

plot_counties_by_cluster(corn_belt_counties, "Corn Belt state clusters", ax_size=(12, 8))